# Bengaluru Flood Resilience & Mobility Decision System
## Module 1 — Flood Intelligence Module (Data Collection → Preprocessing → ML → Evaluation)

This notebook follows the project methodology:

1. **Data Collection** – geospatial & environmental, drainage, lakes, infrastructure, population, historical flood locations, rainfall events
2. **Preprocessing & Integration** – cleaning, common coordinate system (local km grid), ~1 km × 1 km cells, feature engineering
3. **Module 1: Flood Intelligence** – Random Forest / Gradient Boosting (XGBoost-style) with hyper-parameter tuning → flood probability, severity, confidence score
4. **Output & Evaluation** – accuracy, precision, recall, F1, ROC-AUC, cross-validation, validation against real BBMP/KSNDMC hotspots, sensitivity analysis

> **Data note:** real Bengaluru reference points (flood hotspots, lakes, valleys, hospitals) anchor the dataset; terrain, drainage capacity and per-storm flood outcomes are physically simulated because those layers aren't openly published at this resolution. Use `src/real_data_loaders.py` to swap in downloaded files — see `DATA_SOURCES.md`.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('src'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option('display.max_columns', 40)

## Step 1 — Data Collection
Build the city grid with all static layers, and the rainfall-event table with flood labels.

In [ ]:
import build_dataset
build_dataset.main()
grid = pd.read_csv('data/processed/grid_cells.csv')
events = pd.read_csv('data/processed/flood_events.csv', parse_dates=['date'])
print(grid.shape, events.shape)
grid.head()

In [ ]:
from reference_data import FLOOD_HOTSPOTS, LAKES, HOSPITALS
print(len(FLOOD_HOTSPOTS), 'real flood hotspots |', len(LAKES), 'lakes |', len(HOSPITALS), 'hospitals')
pd.DataFrame(FLOOD_HOTSPOTS, columns=['location','lat','lon','severity']).head(10)

## Exploratory Data Analysis

In [ ]:
grid.describe().T[['mean','std','min','max']].round(2)

In [ ]:
df = events.merge(grid, on='cell_id')
print('Overall flood rate:', round(df.flooded.mean(), 3))
rate = df.groupby(pd.cut(df.rain_24h_mm, [0,20,50,80,120,160,250]), observed=True).flooded.mean()
ax = rate.plot.bar(color='#2563eb', figsize=(6,3), title='Share of cells flooded by 24 h rainfall')
ax.set_ylabel('Flood rate'); ax.set_xlabel('24 h rainfall (mm)'); plt.xticks(rotation=0); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, col, cmap in zip(ax, ['elevation_m','impervious_frac','hist_flood_count_2015_24'], ['terrain','Greys','Reds']):
    sc = a.scatter(grid.lon, grid.lat, c=grid[col], cmap=cmap, s=18, marker='s'); a.set_title(col); a.set_aspect('equal')
    plt.colorbar(sc, ax=a, shrink=0.8)
plt.tight_layout(); plt.show()

## Step 2 — Preprocessing & Feature Engineering
Derived risk indicators: `excess_over_drain_mm_hr` (peak rain beyond drain capacity, adjusted for silt), `runoff_index`, `lake_overflow_index`, `monsoon_peak`, and one-hot land use.

In [ ]:
from features import add_features, feature_columns
X = add_features(df)
X[feature_columns()].head()

## Step 3 — Module 1: Train & tune models
* Models: Logistic Regression (baseline), Random Forest, Gradient Boosting (uses XGBoost automatically if installed)
* Tuning: `RandomizedSearchCV` with **GroupKFold by storm** (the model is always scored on storms it has never seen)
* Test set: the **most recent 20% of storms** (time-based hold-out — no leakage from the future)
* Takes ~2 minutes

In [ ]:
import train
train.main()

## Step 4 — Output & Evaluation

In [ ]:
m = json.load(open('outputs/metrics.json'))
print('Best model:', m['best_model'])
pd.read_csv('outputs/model_comparison.csv', index_col=0)

In [ ]:
display(Image('outputs/figures/01_roc_pr_curves.png'))
display(Image('outputs/figures/02_confusion_matrices.png'))

In [ ]:
display(Image('outputs/figures/03_feature_importance.png'))
display(Image('outputs/figures/05_rainfall_response.png'))

### Flood-risk maps & zone ranking

In [ ]:
display(Image('outputs/figures/04_risk_maps_scenarios.png'))
display(Image('outputs/figures/06_zone_ranking.png'))

### Validation against real flood locations & sensitivity analysis

In [ ]:
print(f"Real BBMP/KSNDMC hotspots inside the model's top-10% risk cells (100 mm): {m['hotspot_recall_top10pct_100mm']:.0%}  (random would be 10%)")
print(f"Same check using only physical features (no history): {m['hotspot_recall_top10pct_100mm_no_history']:.0%}")
print('Ablation - test PR-AUC without historical-flood features:', round(m['ablation']['pr_auc_without_history'], 3))
pd.DataFrame(m['severity_report']).T.round(3)

## Module 6 hook — What-if simulator
Rainfall + interventions (drain upgrade, desilting, green cover) → new risk. This is the function the other modules / dashboard call.

In [ ]:
from predict import FloodPredictor
fp = FloodPredictor()
base = fp.scenario(150)
fixed = fp.scenario(150, drain_capacity_boost_pct=30, desilting=True)
print('Cells predicted to flood :', base.flood_predicted.sum(), '->', fixed.flood_predicted.sum())
print('Expected people at risk  :', f"{base.people_at_risk.sum():,.0f} -> {fixed.people_at_risk.sum():,.0f}")
fixed.sort_values('flood_probability', ascending=False).head(10)

In [ ]:
risk = pd.read_csv('outputs/cell_risk_scores.csv')   # hand-off file for Modules 2-7
risk.head()